In [ ]:
from AlgorithmImports import *
from QuantConnect.Research import QuantBook
from QuantConnect.Data.UniverseSelection import FutureUniverse
from datetime import datetime
import pandas as pd
import numpy as np


DATA MAPPING MODE

LAST_TRADING_DAY: DataMappingMode
The contract maps on the previous day of expiration of the front month (0)

FIRST_DAY_MONTH: DataMappingMode
The contract maps on the first date of the delivery month of the front month. If the contract expires prior to this date, then it rolls on the contract's last trading date instead (1)

OPEN_INTEREST: DataMappingMode
The contract maps when the following back month contract has a higher open interest that the current front month (2)

OPEN_INTEREST_ANNUAL: DataMappingMode
The contract maps when any of the back month contracts of the next year have a higher volume that the current front month (3)

DATA NORMALIZATION

FORWARD_PANAMA_CANAL: DataNormalizationMode
Eliminates price jumps between two consecutive contracts, adding a factor based on the difference of their prices. The first contract has the true price. Factor 0. (4)

BACKWARDS_PANAMA_CANAL: DataNormalizationMode
Eliminates price jumps between two consecutive contracts, adding a factor based on the difference of their prices. The last contract has the true price. Factor 0. (5)

BACKWARDS_RATIO: DataNormalizationMode
Eliminates price jumps between two consecutive contracts, multiplying the prices by their ratio. The last contract has the true price. Factor 1. (6)

In [ ]:
start_date = datetime(2023, 1, 1)
exit_date = datetime(2025, 1, 1)

In [ ]:
qb_org = QuantBook()

future_org = qb_org.add_future(
    Futures.Energy.CRUDE_OIL_WTI,
    resolution=Resolution.DAILY,
    extended_market_hours=True,
    data_mapping_mode=DataMappingMode.OPEN_INTEREST,
    data_normalization_mode=DataNormalizationMode.RAW,
    contract_depth_offset=0
)

history_org = qb_org.history(
    future_org.symbol,
    start=start_date,
    end=exit_date,
    resolution=Resolution.DAILY
)

print(history_org.to_string())

In [ ]:
qb = QuantBook()

future = qb.add_future(
    Futures.Energy.CRUDE_OIL_WTI
)

future.set_filter(0, 90)# contracts that expire in 90 days

chain_history = qb.future_history(
    future.symbol,
    start=start_date,
    end=exit_date,
    resolution=Resolution.DAILY
)

contracts = chain_history.data_frame
contracts = contracts.reset_index()

open_interest = qb.history(
    FutureUniverse,
    future.symbol,
    start=start_date,
    end=exit_date,
    flatten=True
).reset_index()

contracts["date"] = contracts["time"].dt.normalize()
open_interest["date"] = open_interest["time"].dt.normalize()

# Not everyday has open interests
contracts = contracts.merge(
    open_interest[["date", "symbol", "openinterest"]],
    on=["date", "symbol"],
    how="left"
)

contracts = contracts.drop(columns="date")

print(contracts.to_string())

In [ ]:
def rollover_org(current_date, current_contract):
    current_date = pd.Timestamp(current_date).normalize()
    current_contracts = contracts[(contracts["time"].dt.normalize() == current_date) & (contracts["expiry"] > current_contract.expiry)].sort_values(by="expiry")

    if current_contracts.empty:
        # non trading day
        return current_contract

    new_contract = contracts[(contracts["time"].dt.normalize() == current_date) & (contracts["symbol"] == current_contract.symbol)].iloc[0]

    # Quant Connect uses same day open interest (not possible as it is posted at the end of day)
    back_month_contract = current_contracts.iloc[0]
    if current_date == current_contract.expiry or (pd.notna(new_contract.openinterest) and back_month_contract.openinterest >= new_contract.openinterest):
        new_contract= current_contracts[current_contracts["symbol"] == back_month_contract.symbol].iloc[0]

    # Quant Connect looks ahead for open interest if current day open interest is not reported
    if pd.isna(new_contract.openinterest):
        next_date=current_date+ pd.Timedelta(days=1)
        next_contracts = contracts[(contracts["time"].dt.normalize() == next_date) & (contracts["expiry"] > current_contract.expiry)].sort_values(by="expiry")

        if next_contracts.empty:
            # If no immediate next day keep current contract
            return new_contract

        next_current_contract = contracts[(contracts["time"].dt.normalize() == next_date) & (contracts["symbol"] == current_contract.symbol)].iloc[0]
        next_back_month_contract = next_contracts.iloc[0]
        if next_back_month_contract.openinterest >= next_current_contract.openinterest:
            new_contract= current_contracts[current_contracts["symbol"] == back_month_contract.symbol].iloc[0]

    return new_contract

In [ ]:
def rollover(current_date, current_contract):
    current_date = pd.Timestamp(current_date).normalize()
    current_contracts = contracts[contracts["time"].dt.normalize() == current_date]

    if current_contracts.empty:
        # non trading day
        return current_contract

    new_contract = current_contracts[current_contracts["symbol"] == current_contract.symbol].iloc[0]

    previous_dates = contracts.loc[contracts["time"] < current_date]

    if previous_dates.empty:
        # first trading day
        return new_contract

    # Use last trading days data to decide rollover
    prev_date = previous_dates["time"].max().normalize()

    prev_contracts = contracts[(contracts["time"].dt.normalize() == prev_date) & (contracts["expiry"] > current_contract.expiry)].sort_values(by="expiry")

    if prev_contracts.empty:
        print("should not happen")
        return new_contract

    # OPEN_INTEREST on just next month
    back_month_contract = prev_contracts.iloc[0]
    if current_date == current_contract.expiry or (pd.notna(current_contract.openinterest) and back_month_contract.openinterest >= current_contract.openinterest):
        new_contract= current_contracts[current_contracts["symbol"] == back_month_contract.symbol].iloc[0]

    return new_contract

In [ ]:
def simulate (start_date, exit_date):
    entry_date = pd.Timestamp(start_date).normalize()
    exit_date = pd.Timestamp(exit_date).normalize()

    # Pick entry contract
    while True:
        entry_contracts = contracts[contracts["time"].dt.normalize() == entry_date]
        if not entry_contracts.empty:
            break
        print("no data on day: ", entry_date, "picking next day")
        entry_date += pd.Timedelta(days=1)

    # Choose earliest max volume contract
    max_volume = entry_contracts["volume"].max()
    highest_volume_contracts = entry_contracts[entry_contracts["volume"] == max_volume]
    highest_volume_contract = highest_volume_contracts.iloc[0]

    print("Picking contract with symbol: ", highest_volume_contract.symbol, "expiry: ", highest_volume_contract.expiry)

    # data on each day
    rows=[]
    current_contract = highest_volume_contract
    for current_date in pd.date_range(entry_date, exit_date, inclusive="left"):
        current_contract = rollover(current_date, current_contract)

        row = current_contract.to_dict()  # copies every column
        row["current_date"] = current_date
        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
data = simulate(start_date, exit_date)

print(data.to_string())

In [ ]:
def compare(impl, org):
    columns_to_compare = [
        "askclose", "askhigh", "asklow", "askopen", "asksize",
        "bidclose", "bidhigh", "bidlow", "bidopen", "bidsize",
        "close", "high", "low", "open", "volume"
    ]

    impl = impl.copy()
    org = org.copy()

    # Restore index columns if necessary
    if "current_date" not in impl.columns and "time" not in impl.columns:
        impl = impl.reset_index()

    if "time" not in org.columns:
        org = org.reset_index()

    # impl uses current_date; org uses time
    if "current_date" in impl.columns:
        impl_date_column = "current_date"
    elif "time" in impl.columns:
        impl_date_column = "time"
    else:
        raise KeyError("impl must contain either 'current_date' or 'time'")

    if "time" not in org.columns:
        raise KeyError("org must contain a 'time' column")

    # Compare calendar dates rather than exact timestamps
    impl["_compare_date"] = pd.to_datetime(
        impl[impl_date_column]
    ).dt.normalize()

    org["_compare_date"] = pd.to_datetime(
        org["time"]
    ).dt.normalize()

    # Unique ID prevents duplicate org rows from duplicating final output
    impl["_impl_row_id"] = np.arange(len(impl))

    merged = impl.merge(
        org,
        on="_compare_date",
        how="left",
        suffixes=("_impl", "_org"),
        indicator=True
    )

    org_row_exists = merged["_merge"].eq("both")

    values_match = np.logical_and.reduce([
        np.isclose(
            merged[f"{column}_impl"],
            merged[f"{column}_org"],
            equal_nan=True
        )
        for column in columns_to_compare
    ])

    # A candidate matches only when org actually has that row
    merged["_candidate_match"] = org_row_exists & values_match

    # Reduce multiple org rows back to one result per impl row
    result = (
        merged.groupby("_impl_row_id", sort=False)
        .agg(
            org_exists=("_merge", lambda values: values.eq("both").any()),
            values_match=("_candidate_match", "any")
        )
        .reset_index()
    )

    # Missing from org counts as True
    result["same"] = (
        ~result["org_exists"]
        | result["values_match"]
    )

    # Restore the original impl date/time for display
    result = result.merge(
        impl[["_impl_row_id", impl_date_column]],
        on="_impl_row_id",
        how="left"
    )

    result = result[
        [impl_date_column, "org_exists", "same"]
    ].rename(columns={impl_date_column: "time"})

    print(result.to_string(index=False))
    print("\nAll true:", result["same"].all())



In [ ]:
# Quant Connect decides rollovers based on same day open interest, and also when open interest is NaN, it probably uses next days open interest to determine a skip
compare(data, history_org)